# MUSIC 데이터 확인과 기본 모델 비교

위에서부터 셀을 실행한다. **CSV·코드북 확인 → 공통 환자 분할 확인 → 기본 모델 학습 → 불균형·평가지표 비교** 순서다.

- 터미널에서 `dev music`, `dev init`, `dev practice`를 실행한 뒤 생성된 개인 노트북을 연다.
- MUSIC 1.0.1의 세 CSV와 `music-scd-basic-v1` 공통 데이터를 사용한다. 파형 다운로드는 필요 없다.
- 학습 데이터로만 전처리·모델을 학습하고, 검증 데이터로 비교한다. 테스트 데이터의 예측·평가는 하지 않는다.
- 표·그림은 개인 실습 폴더에 저장한다. MLflow 전송과 제출용 모델 저장은 마지막의 개인 `train.py` 흐름에서 진행한다.

## 1. 실행 위치와 실습 설정

`FEATURE_SET`은 `clinical`(임상정보), `ecg`(심전도 특징), `combined`(둘 다) 중 하나다. 먼저 기본값으로 끝까지 실행한 뒤, 바꾸고 전체 셀을 다시 실행해 본다. 같은 실행 안의 세 모델은 항상 같은 환자·같은 입력을 사용한다.

In [ ]:
import hashlib
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sixletters.data.music import SOURCE_FILES
from sixletters.data.prepared import load_prepared_dataset
from sixletters.local_settings import project_root
from sixletters.workspace import validate_component

OWNER = "__POC_OWNER__"
FEATURE_SET = "clinical"
SEED = 42
THRESHOLD = 0.5
ROOT = project_root(Path.cwd())
RAW_DIR = ROOT / "Global/data/raw/music-1.0.1"
DATA_DIR = ROOT / "Global/data/prepared/music-scd-basic-v1"
if OWNER == "__POC_OWNER__":
    raise ValueError("터미널에서 dev practice로 개인 노트북을 생성한 뒤 실행하세요.")
OWNER = validate_component(OWNER, "owner")
if FEATURE_SET not in {"clinical", "ecg", "combined"}:
    raise ValueError("FEATURE_SET은 clinical, ecg, combined 중 하나여야 합니다.")
OUTPUT_DIR = ROOT / "Global/artifacts" / OWNER / "practice" / FEATURE_SET
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {
        "figure.figsize": (9, 4.5),
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "savefig.dpi": 150,
    }
)
# 그래프는 운영체제별 한글 폰트 설치 없이 읽을 수 있도록 영문 표기를 사용한다.
print(f"실습 결과: {OUTPUT_DIR.relative_to(ROOT)}")

## 2. 공식 설명과 세 CSV 확인

[MUSIC 공식 페이지](https://physionet.org/content/music-sudden-cardiac-death/1.0.1/)의 **Abstract / Data Description / Usage Notes**를 읽으며 다음을 확인한다.

| 확인할 내용 | 실습에서 사용할 기준 |
|---|---|
| 어떤 환자 자료인가? | MUSIC 연구의 만성 심부전 환자 자료 |
| 어떤 파일이 있는가? | 환자 정보 CSV, 범주형 코드 설명 CSV, 약어 설명 CSV |
| 모든 환자의 관찰 기간이 같은가? | 서로 다르므로 관찰 중 발생한 사건과 특정 기간 내 위험을 구분 |
| CSV와 파형은 어떻게 다른가? | 이 실습은 CSV의 임상정보·심전도 특징을 사용하며, 원시 파형은 다루지 않음 |

`dev music`이 받은 파일의 해시(파일 내용의 식별값)를 다시 검사한다. 파일 누락·변경 시 임의로 수정하지 말고 오류를 확인한다.

In [ ]:
source_rows = []
for filename, expected_hash in SOURCE_FILES.items():
    path = RAW_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"{filename}이 없습니다. 터미널에서 dev music을 실행하세요.")
    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f"{filename}이 공통 원본과 다릅니다. 파일 변경 여부를 확인하세요.")
    source_rows.append({"파일": filename, "크기(bytes)": path.stat().st_size, "해시 확인": "일치"})
source_summary = pd.DataFrame(source_rows)
display(source_summary)
print(f"세 CSV 합계: {source_summary['크기(bytes)'].sum() / 1_000_000:.3f} MB")

### CSV를 올바른 형식으로 읽기

세 파일은 쉼표가 아닌 **세미콜론(`;`)으로 열을 구분**한다. 환자 CSV는 소수점에 쉼표를 사용한다(`decimal=","`). 코드북은 `cp1252`로 읽는다.

원본 열 이름의 공백은 유지한다. 예를 들어 `Average RR (ms)`와 `Average RR (ms) `는 서로 다른 측정 열이므로 모든 열 이름에 `.strip()`을 적용하면 구분이 사라진다.

In [ ]:
subject = pd.read_csv(
    RAW_DIR / "subject-info.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    dtype={"Patient ID": "string"},
).replace(r"^\s*$", np.nan, regex=True)
codes = pd.read_csv(
    RAW_DIR / "subject-info_codes.csv",
    sep=";",
    encoding="cp1252",
    dtype=str,
)
definitions = pd.read_csv(
    RAW_DIR / "subject-info_definitions.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
)
# 코드북의 빈 Variable/Column은 바로 위 변수에 대한 설명이 이어지는 행이다.
codes[["Variable", "Column"]] = codes[["Variable", "Column"]].ffill()
display(
    pd.DataFrame(
        [
            {"파일": "subject-info.csv", "행": subject.shape[0], "열": subject.shape[1]},
            {"파일": "subject-info_codes.csv", "행": codes.shape[0], "열": codes.shape[1]},
            {
                "파일": "subject-info_definitions.csv",
                "행": definitions.shape[0],
                "열": definitions.shape[1],
            },
        ]
    )
)
assert subject.shape == (992, 103), "MUSIC 1.0.1 원본 크기와 다릅니다."
assert subject["Patient ID"].notna().all()
assert subject["Patient ID"].is_unique
print("환자 식별키: Patient ID / 누락 0건 / 중복 0건")

### 열별 자료형과 결측률

결측률은 해당 열의 빈 값 수를 전체 환자 수로 나눈 비율이다. **빈 값은 0과 다르다.** 자료형이 숫자라도 0·1·3 같은 범주 코드일 수 있으므로 자료형만 보고 판단하지 않는다.

`Exit of the study`의 빈 값은 `Cause of death=0`인 기록에서 나타난다. 이를 모든 환자의 정답 누락으로 처리하거나, 결측률이 높은 행을 일괄 삭제하지 않는다. 원본 연령의 `>89` 같은 표기도 공통 준비 기준에서 처리한다.

In [ ]:
column_summary = pd.DataFrame(
    {
        "원본 열": subject.columns,
        "자료형": subject.dtypes.astype(str).to_numpy(),
        "결측 수": subject.isna().sum().to_numpy(),
        "결측률(%)": subject.isna().mean().mul(100).to_numpy(),
        "서로 다른 값 수": subject.nunique(dropna=True).to_numpy(),
    }
)
column_summary.to_csv(OUTPUT_DIR / "raw_column_summary.csv", index=False)
display(column_summary.sort_values("결측률(%)", ascending=False).head(12).round(2))
print("전체 103개 열의 자료형·결측률은 raw_column_summary.csv에서 확인합니다.")

missing_top = subject.isna().mean().mul(100).sort_values(ascending=False).head(8).sort_values()
fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
ax.barh(missing_top.index, missing_top.values, color="#33658A")
ax.set(
    xlabel="Missing values (% of 992 patients)",
    xlim=(0, 100),
    title="MUSIC 1.0.1: fields with the most missing values",
)
ax.grid(axis="x", alpha=0.2)
fig.savefig(OUTPUT_DIR / "missing_values.png", bbox_inches="tight")
plt.show()

### 범주 코드와 약어의 의미

코드북의 `Column`은 원본 CSV의 엑셀식 열 위치다. 열 이름만 유사하게 맞추지 않고 이 위치로 연결한다.

코드북은 일부 범주형 변수만, 약어 문서는 일부 약어만 설명한다. **103개 열 모두에 대한 완전한 임상 정의서는 아니다.** 설명이 없는 변수의 뜻·단위를 추측해서 사용하지 않는다.

In [ ]:
def excel_position(label):
    position = 0
    for character in label:
        position = position * 26 + ord(character) - ord("A") + 1
    return position - 1


codebook_columns = codes[["Variable", "Column"]].drop_duplicates().copy()
codebook_columns["원본 열"] = codebook_columns["Column"].map(
    lambda value: subject.columns[excel_position(value)]
)
codebook_columns["관측된 값 종류 수"] = codebook_columns["원본 열"].map(
    lambda value: subject[value].nunique(dropna=True)
)
display(codebook_columns.reset_index(drop=True))
display(definitions)

In [ ]:
# COLUMN을 다른 코드북 변수 이름으로 바꾸면 같은 방식으로 분포와 뜻을 확인할 수 있다.
COLUMN = "HF etiology - Diagnosis"
original_column = codebook_columns.set_index("Variable").loc[COLUMN, "원본 열"]
display(codes.loc[codes["Variable"].eq(COLUMN), ["Variable", "Codes"]].reset_index(drop=True))
category_counts = subject[original_column].value_counts(dropna=False).sort_index()
display(category_counts.rename_axis("원본 코드").to_frame("환자 수"))

### 사망원인 코드와 예측 정답 구분

**심부전 환자인지를 분류하는 실습이 아니다.** 공통 `basic-v1`은 연구의 관찰 기간 중 돌연심장사(SCD)가 기록되었는지 비교하는 이진 분류 기준이다.

- `Cause of death=3`을 1, 그 외 포함 환자를 0으로 정한다.
- 추적 중단(`Exit=1`)·심장이식(`Exit=2`) 환자는 공통 기준으로 제외한다.
- 0에는 생존자뿐 아니라 다른 원인으로 사망한 환자도 포함된다.
- 관찰 기간이 다르므로 **고정된 4년 내 사망 위험 예측을 뜻하지 않는다.** 최종 예측 대상·기간은 기업과 합의해야 한다.

In [ ]:
display(
    codes.loc[
        codes["Variable"].isin(["Exit of the study", "Cause of death"]), ["Variable", "Codes"]
    ].reset_index(drop=True)
)
cause_meaning = {
    0: "생존으로 기록",
    1: "비심장성 사망",
    3: "돌연심장사(SCD)",
    6: "심장 펌프 기능 부전 사망",
    7: "심장 펌프 기능 부전 사망",
}
cause_counts = (
    subject["Cause of death"].value_counts().sort_index().rename("원본 환자 수").to_frame()
)
cause_counts["코드 의미"] = cause_counts.index.map(cause_meaning)
display(cause_counts)
print("추적 중단:", int(subject["Exit of the study"].eq(1).sum()), "명")
print("심장이식:", int(subject["Exit of the study"].eq(2).sum()), "명")

## 3. 공통 데이터와 환자 분할 확인

정답·입력 열·분할을 여기서 새로 만들지 않고 `dev music`의 결과를 읽는다. 같은 환자가 학습·검증·테스트에 섞이지 않아야 새로운 환자에 대한 성능을 비교할 수 있다.

| 구분 | 용도 |
|---|---|
| train / 학습 | 결측값을 채울 기준, 표준화 기준, 모델을 학습 |
| validation / 검증 | 학습 방법·설정의 성능 비교 |
| test / 테스트 | 방법을 결정한 뒤 최종 평가. 이 실습에서는 사용하지 않음 |

분할 비율은 약 70:15:15다. 환자 수를 정수로 배정하므로 정확한 비율은 아래 표와 같다.

In [ ]:
if not (DATA_DIR / "dataset.json").is_file():
    raise FileNotFoundError("공통 basic 데이터가 없습니다. 터미널에서 dev music을 실행하세요.")
dataset, synthetic = load_prepared_dataset(DATA_DIR)
assert not synthetic
assert dataset.dataset_id == "music-1.0.1-scd-basic-v1"
frame = dataset.frame
patient_sets = {
    split: set(frame.loc[frame["split"].eq(split), "patient_id"])
    for split in ("train", "validation", "test")
}
assert patient_sets["train"].isdisjoint(patient_sets["validation"])
assert patient_sets["train"].isdisjoint(patient_sets["test"])
assert patient_sets["validation"].isdisjoint(patient_sets["test"])
assert sum(map(len, patient_sets.values())) == len(frame)
split_summary = (
    frame.groupby("split")["target"]
    .agg(patients="size", scd="sum")
    .reindex(["train", "validation", "test"])
)
split_summary["scd_rate"] = split_summary["scd"] / split_summary["patients"]
display(split_summary.round(3))
print("분할 간 같은 환자: 0명 / 공통 대상:", len(frame), "명")
print("공통 정답 정의:", dataset.target_definition)

### 직접 확인: 데이터 누수

검증 데이터까지 합쳐 결측값을 채울 중앙값을 계산하면 왜 문제가 되는가?

**내 답:**



### 클래스 불균형 확인

클래스는 예측할 정답의 종류(0 또는 1)를 뜻한다. 두 종류의 수가 크게 다른 것이 **클래스 불균형**이다. 여기서 양성은 정답 1(SCD), 음성은 정답 0을 뜻한다.

아래 표에서 학습 데이터의 1 비율을 확인한다. 0이 대부분이면 전부 0으로 답해도 정확도가 높아질 수 있다. 그래서 정확도 하나로 모델을 선택하지 않는다.

In [ ]:
features = (
    list(dataset.feature_groups[FEATURE_SET])
    if FEATURE_SET != "combined"
    else [*dataset.feature_groups["clinical"], *dataset.feature_groups["ecg"]]
)
train = frame.loc[frame["split"].eq("train")].set_index("patient_id")
validation = frame.loc[frame["split"].eq("validation")].set_index("patient_id")
X_train, y_train = train[features], train["target"].astype(int)
X_validation, y_validation = validation[features], validation["target"].astype(int)
assert set(y_train.unique()) == {0, 1} and set(y_validation.unique()) == {0, 1}
counts = y_train.value_counts().reindex([0, 1])
imbalance = pd.DataFrame(
    {
        "학습 환자 수": counts,
        "학습 비율": counts / len(y_train),
        "balanced 가중치": len(y_train) / (2 * counts),
    }
)
display(imbalance.round(3))
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
bars = ax.bar(["Target 0", "Target 1 (SCD)"], counts, color=["#33658A", "#B45F06"])
ax.bar_label(bars, padding=3)
ax.set(
    ylabel="Training patients",
    ylim=(0, counts.max() * 1.15),
    title=f"Training class balance (n={len(y_train)})",
)
fig.savefig(OUTPUT_DIR / "training_class_balance.png", bbox_inches="tight")
plt.show()

## 4. 세 가지 기준으로 학습

| 모델 | 확인할 내용 |
|---|---|
| `Dummy prior` | 모든 환자에게 같은 점수(학습 데이터의 양성 비율)를 주는 기준선. 특징을 학습하지 않음 |
| `Logistic` | 로지스틱 회귀 기본 설정 |
| `Logistic balanced` | 같은 로지스틱 회귀에서 적은 클래스의 학습 오류에 더 큰 가중치를 부여 |

`class_weight="balanced"`는 환자 복제나 데이터 분할 변경이 아니다. **양성 환자를 더 잘 찾을 수도 있지만 오탐이 늘거나 다른 지표가 나빠질 수도 있다.** 같은 검증 환자에서 실제 결과를 비교한다.

결측 처리·표준화·범주형 값 변환은 `Pipeline`에 넣는다. `.fit(X_train, y_train)`만 호출하므로 검증 데이터의 통계가 학습에 섞이지 않는다. 전체 데이터로 먼저 결측값을 채우거나 표준화하면 **데이터 누수**가 된다.

In [ ]:
categorical_columns = [column for column in features if column in dataset.categorical_features]
numeric_columns = [column for column in features if column not in categorical_columns]
numeric = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
    ]
)
categorical = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocessing = ColumnTransformer(
    [
        ("numeric", numeric, numeric_columns),
        ("categorical", categorical, categorical_columns),
    ]
)
classifiers = {
    "Dummy prior": DummyClassifier(strategy="prior"),
    "Logistic": LogisticRegression(C=1.0, max_iter=3000, random_state=SEED),
    "Logistic balanced": LogisticRegression(
        C=1.0, class_weight="balanced", max_iter=3000, random_state=SEED
    ),
}
models, validation_scores = {}, {}
for name, classifier in classifiers.items():
    model = Pipeline([("preprocessing", clone(preprocessing)), ("classifier", classifier)])
    model.fit(X_train, y_train)
    models[name] = model
    positive_column = list(model.classes_).index(1)
    validation_scores[name] = model.predict_proba(X_validation)[:, positive_column]
print(
    f"입력: {FEATURE_SET} / {len(features)}개 열 / 학습 {len(y_train)}명 / 검증 {len(y_validation)}명"
)

## 5. 정확도·재현율·정밀도·AUROC·AP 비교

| 지표 | 읽는 방법 |
|---|---|
| 정확도 / Accuracy | 전체 검증 환자 중 정답을 맞힌 비율 |
| 재현율 / Recall | 실제 1인 환자 중 1로 예측한 비율. 놓친 환자가 적을수록 높음 |
| 정밀도 / Precision | 1로 예측한 환자 중 실제 1인 비율. 오탐이 적을수록 높음 |
| AUROC | 실제 1인 환자를 실제 0인 환자보다 높은 점수에 놓는 능력. 동일 점수 기준선은 0.5 |
| AP / Average Precision | 양성 환자를 얼마나 잘 찾으면서 오탐을 줄이는지 PR 관계를 요약. 동일 점수 기준선은 검증 양성 비율 |
| PR-AUC (trapezoid) | PR 곡선 점들을 직선으로 이어 면적을 계산. **AP와 계산법이 달라 같은 값이 아님** |

정확도·재현율·정밀도는 점수가 `THRESHOLD=0.5` 이상이면 1로 예측해서 계산한다. AUROC·AP·PR-AUC는 점수 전체를 사용한다. 임계값 0.5는 실습용이며 의료적 판단 기준이 아니다.

1로 예측한 환자가 없으면 정밀도를 나눗셈으로 계산할 수 없어 표에서는 0으로 표시한다. AP와 PR 면적의 차이는 [scikit-learn 공식 설명](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)을 참고한다.

In [ ]:
metric_rows = []
for name, scores in validation_scores.items():
    predicted = (scores >= THRESHOLD).astype(int)
    precision_curve, recall_curve, _ = precision_recall_curve(y_validation, scores)
    metric_rows.append(
        {
            "model": name,
            "predicted_positive": int(predicted.sum()),
            "accuracy": accuracy_score(y_validation, predicted),
            "recall": recall_score(y_validation, predicted, zero_division=0),
            "precision": precision_score(y_validation, predicted, zero_division=0),
            "roc_auc": roc_auc_score(y_validation, scores),
            "average_precision": average_precision_score(y_validation, scores),
            "pr_auc_trapezoid": auc(recall_curve[::-1], precision_curve[::-1]),
        }
    )
metrics = pd.DataFrame(metric_rows).set_index("model")
display(metrics.round(3))
validation_prevalence = float(y_validation.mean())
print(f"검증 양성: {int(y_validation.sum())}/{len(y_validation)}명 ({validation_prevalence:.1%})")
print(f"전부 0으로 답했을 때 정확도: {1 - validation_prevalence:.1%}")
print(f"동일 점수 기준선 AUROC: 0.500 / AP: {validation_prevalence:.3f}")
assert np.isclose(metrics.loc["Dummy prior", "roc_auc"], 0.5)
assert np.isclose(metrics.loc["Dummy prior", "average_precision"], validation_prevalence)

**표에서 확인할 것**

- `Dummy prior`는 1을 하나도 찾지 못하는데도 정확도가 높은가? `predicted_positive`와 `recall`을 함께 본다.
- `balanced`의 재현율이 높아졌다면 정밀도·AP는 어떻게 달라졌는가? 어느 지표든 개선을 가정하지 않는다.
- 상수 점수 기준선의 PR-AUC는 끝점 연결 방식 때문에 AP보다 크게 나올 수 있다. 이를 학습 능력이 생긴 것으로 해석하지 않는다. **공통 실험 비교의 PR 계열 지표는 `average_precision`(AP)이다.**

### 직접 비교: 정확도와 불균형 처리

1. `Dummy prior`의 정확도와 재현율은 각각 얼마인가? 이 모델을 정확도만 보고 선택하면 무엇을 놓치는가?

**내 답:**

2. `Logistic balanced`는 기본 모델보다 놓친 양성 환자·오탐·AP가 각각 어떻게 달라졌는가? 아래 혼동행렬까지 확인하고 답한다.

**내 답:**



### ROC 곡선과 PR 곡선

ROC의 가로축은 실제 0인 환자를 1로 잘못 예측한 비율, 세로축은 재현율이다. PR은 가로축이 재현율, 세로축이 정밀도다. 두 그림 모두 같은 검증 환자를 사용한다.

ROC에서는 대각선, PR에서는 검증 양성 비율 수평선을 기준으로 본다. PR 그림은 단계선으로 표시하며, 앞 표의 사다리꼴 PR 면적과 구분한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), layout="constrained")
colors = {"Dummy prior": "#666666", "Logistic": "#33658A", "Logistic balanced": "#B45F06"}
line_styles = {"Dummy prior": ":", "Logistic": "-", "Logistic balanced": "--"}
for name, scores in validation_scores.items():
    fpr, tpr, _ = roc_curve(y_validation, scores)
    axes[0].plot(
        fpr,
        tpr,
        color=colors[name],
        linestyle=line_styles[name],
        label=f"{name}: AUROC={metrics.loc[name, 'roc_auc']:.3f}",
    )
    if name != "Dummy prior":
        precision_curve, recall_curve, _ = precision_recall_curve(y_validation, scores)
        axes[1].step(
            recall_curve,
            precision_curve,
            where="post",
            color=colors[name],
            linestyle=line_styles[name],
            label=f"{name}: AP={metrics.loc[name, 'average_precision']:.3f}",
        )
axes[1].axhline(
    validation_prevalence,
    color=colors["Dummy prior"],
    linestyle=":",
    label=f"Constant-score AP baseline={validation_prevalence:.3f}",
)
axes[0].set(
    xlabel="False positive rate",
    ylabel="Recall (true positive rate)",
    title=f"Validation ROC (n={len(y_validation)})",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
axes[1].set(
    xlabel="Recall",
    ylabel="Precision",
    title="Validation precision-recall",
    xlim=(0, 1),
    ylim=(0, 1.02),
)
for ax in axes:
    ax.legend(loc="upper right" if ax is axes[1] else "lower right", fontsize=9)
    ax.grid(alpha=0.2)
fig.savefig(OUTPUT_DIR / "validation_roc_pr.png", bbox_inches="tight")
plt.show()

### 오탐·놓친 환자를 실제 인원으로 확인

혼동행렬의 **행은 실제 정답**, **열은 예측**이다. 오른쪽 위는 오탐(실제 0 → 예측 1), 왼쪽 아래는 놓친 양성 환자(실제 1 → 예측 0)다. 비율 차이를 인원수와 함께 확인한다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout="constrained")
for ax, (name, scores) in zip(axes, validation_scores.items(), strict=True):
    matrix = confusion_matrix(y_validation, (scores >= THRESHOLD).astype(int), labels=[0, 1])
    ConfusionMatrixDisplay(matrix, display_labels=[0, 1]).plot(
        ax=ax,
        colorbar=False,
        cmap="Blues",
        values_format="d",
        im_kw={"vmin": 0, "vmax": len(y_validation)},
    )
    ax.set_title(name)
fig.suptitle(f"Validation patients at threshold {THRESHOLD:g}")
fig.savefig(OUTPUT_DIR / "validation_confusion_matrices.png", bbox_inches="tight")
plt.show()

## 6. 결과 확인과 개인 실험으로 이어가기

아래 셀은 **집계표·그래프·실습 설정만 로컬에 저장**한다. 같은 입력 종류로 다시 실행하면 해당 실습 파일을 갱신한다. 환자별 원본·예측 결과·학습 모델은 이 실습에서 내보내지 않는다.

검증 데이터의 양성 환자 수가 적어 몇 명의 예측 차이로 점수가 크게 달라질 수 있다. 이 한 번의 결과로 우열을 확정하지 않는다. 검증 결과를 보며 반복 수정한 모델의 최종 성능은 별도 테스트 평가가 필요하다.

In [ ]:
metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv")
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv")
summary = {
    "owner": OWNER,
    "purpose": "music_practice",
    "feature_set": FEATURE_SET,
    "features": features,
    "seed": SEED,
    "threshold": THRESHOLD,
    "dataset_fingerprint": dataset.fingerprint(),
    "source_sha256": SOURCE_FILES,
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "model_settings": {
        name: classifier.get_params(deep=False) for name, classifier in classifiers.items()
    },
    "test_scored": False,
    "mlflow_uploaded": False,
}
(OUTPUT_DIR / "practice_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("완료:", OUTPUT_DIR.relative_to(ROOT))
print("파일:", ", ".join(path.name for path in sorted(OUTPUT_DIR.iterdir()) if path.is_file()))

### 직접 정리: 다음 실험

1. AP와 사다리꼴 PR-AUC 값이 다른데 두 값을 같은 지표로 비교해도 되는가?

**내 답:**

2. 다음 실험에서 바꿀 설정 한 가지와 그대로 유지할 데이터·분할·평가 기준을 적는다.

**내 답:**



### 완료 확인

- [ ] 세 CSV의 역할, 원본 992행·103열, 환자 ID 중복 여부를 확인했다.
- [ ] 결측률과 범주 코드를 확인했고, 빈 값과 0을 구분할 수 있다.
- [ ] 정답 1과 0에 어떤 환자가 포함되는지 설명할 수 있다.
- [ ] 전처리·모델의 `fit`은 학습 데이터에만 적용되는 것을 확인했다.
- [ ] 기본 모델과 `balanced`의 놓친 양성 환자 수·오탐 수를 비교했다.
- [ ] 정확도가 높아도 재현율이 0일 수 있음을 확인했고, AUROC·AP·사다리꼴 PR-AUC를 구분한다.

### 다음 개인 실험

Docker 작업 환경을 켜 둔 상태에서 **Ubuntu 터미널의 저장소 폴더**에서 실행한다. macOS는 macOS 터미널을 사용한다. 이미 `exp001`이 있으면 새 실험명을 쓴다.

```sh
./dev new exp001 --feature-set clinical
./dev exec exp001 train.py
```

생성된 `Members/이름/experiments/exp001/train.py`에서 전처리·모델을 수정하고 같은 `./dev exec` 명령으로 다시 실행한다. 파일명을 바꿨다면 명령에도 바뀐 이름을 쓴다. JupyterLab의 **Launcher → Terminal**에서는 `./dev` 대신 `dev`를 사용한다. 이 명령을 노트북의 Python 셀에 입력하지 않는다.

- 데이터·환자 분할은 공통 기준을 유지한다.
- 정식 실험은 개인 `train.py`의 `Experiment.save()`로 저장·비교한다. MLflow 주소가 설정되어 있으면 공통 전송 흐름을 사용한다.
- 이 노트북에서 점수가 좋았다는 이유만으로 해당 모델을 최종 선택하지 않는다. 바꾼 설정과 검증 결과를 개인 실험 기록으로 남긴다.
- 노트북의 실행 결과에는 데이터 집계가 포함된다. Git에 올릴 때는 기존 규칙대로 출력값을 제거한다.

### 참고 문서

- [MUSIC 1.0.1: Abstract / Data Description / Usage Notes](https://physionet.org/content/music-sudden-cardiac-death/1.0.1/)
- [scikit-learn: 학습 데이터와 평가 데이터 분리·누수 방지](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)
- [scikit-learn: LogisticRegression의 class_weight](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [scikit-learn: Average Precision과 PR 면적의 차이](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)
- [scikit-learn: 분류 평가지표](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)